# Análisis de resultados — TP NLP Grupo 2

Lee lo que generó `src/run_eval.py` en `results/<split>/`. No llama a ninguna API.

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import config

SPLIT = 'test'
R = config.RESULTS_DIR / SPLIT
pd.set_option('display.max_colwidth', 200)

## 1. Dataset y revisión humana

In [ ]:
ds = pd.read_csv(config.DATA_DIR / 'dataset_revisado.csv') if (config.DATA_DIR / 'dataset_revisado.csv').exists() else pd.read_csv(config.DATASET_PATH)
display(pd.crosstab(ds['categoria_gold'], ds['dificultad'], margins=True))
rev = config.RESULTS_DIR / 'revision_humana.json'
if rev.exists():
    print(json.dumps(json.loads(rev.read_text()), indent=2))

## 2. Resumen por sistema

In [ ]:
summary = pd.read_csv(R / 'summary.csv')
summary[['system', 'model', 'accuracy', 'accuracy_ci95', 'macro_f1', 'macro_f1_lo', 'macro_f1_hi', 'invalid_or_missing_rate', 'latency_p50_ms', 'latency_p95_ms', 'usd_per_1000_equiv']]

In [ ]:
display(Image(R / 'figures' / 'quality_vs_cost.png'))
display(Image(R / 'figures' / 'quality_vs_latency.png'))

## 3. ¿Las diferencias son significativas? (McNemar, Holm)

In [ ]:
pd.read_csv(R / 'mcnemar.csv') if (R / 'mcnemar.csv').exists() else 'Se necesitan al menos 2 sistemas'

## 4. Por categoría y matrices de confusión

In [ ]:
pc = pd.read_csv(R / 'per_class.csv')
display(pc.pivot(index='categoria', columns='system', values='f1').round(3))
display(Image(R / 'figures' / 'confusion.png'))

## 5. Failure modes: dificultad y multi-intención

In [ ]:
bd = pd.read_csv(R / 'by_difficulty.csv')
display(bd.pivot(index='system', columns='dificultad', values='accuracy'))
display(Image(R / 'figures' / 'by_difficulty.png'))
display(pd.read_csv(R / 'multi_intent.csv'))

## 6. Routing en cascada y calibración de TF-IDF

In [ ]:
if (R / 'cascade.csv').exists():
    cas = pd.read_csv(R / 'cascade.csv')
    display(Image(R / 'figures' / 'cascade.png'))
    # Best threshold per LLM: the highest macro-F1, and among ties the one that sends the least to the LLM
    best = cas.sort_values(['macro_f1', 'llm_share'], ascending=[False, True]).drop_duplicates('llm_system')
    display(best)
if (R / 'figures' / 'calibration_tfidf.png').exists():
    display(Image(R / 'figures' / 'calibration_tfidf.png'))

## 7. Análisis cualitativo de errores

Completar la columna `tipo_error` en `results/test/errores/*.csv` (por ejemplo: `ignora_prelacion`, `jerga`, `contexto_confundido`, `ambigua_irresoluble`, `error_de_etiqueta`) y volver a ejecutar esta celda.

In [ ]:
err_dir = R / 'errores'
for path in sorted(err_dir.glob('*.csv')):
    df = pd.read_csv(path, sep=';', encoding='utf-8-sig', keep_default_na=False)
    print(f'{path.stem}: {len(df)} casos')
    if 'tipo_error' in df and (df['tipo_error'] != '').any():
        display(df['tipo_error'].value_counts())
display(pd.read_csv(err_dir / 'fallan_todos.csv', sep=';', encoding='utf-8-sig').head(10))

## 8. (Opcional) Robustez ante ruido

In [ ]:
P = config.RESULTS_DIR / 'perturbed' / 'summary.csv'
if P.exists():
    pert = pd.read_csv(P)[['system', 'accuracy', 'macro_f1']]
    base = pd.read_csv(R / 'by_difficulty.csv').query("dificultad == 'facil'")[['system', 'accuracy']]
    display(base.merge(pert, on='system', suffixes=('_facil_original', '_perturbado')))